In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Створення менеджера проходів для динамічного розв'язування

<Accordion>
<AccordionItem title="Версії пакетів">

Код на цій сторінці розроблено з використанням наведених нижче залежностей.
Рекомендуємо використовувати ці або новіші версії.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Ця сторінка демонструє, як використовувати прохід [`PadDynamicalDecoupling`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.PadDynamicalDecoupling), щоб додати до схеми техніку придушення помилок під назвою _динамічне розв'язування_.

Динамічне розв'язування працює шляхом додавання послідовностей імпульсів (відомих як _послідовності динамічного розв'язування_) до простоюючих кубітів, щоб перевернути їх навколо сфери Блоха. Це скасовує вплив каналів шуму та пригнічує декогеренцію. Такі імпульсні послідовності схожі на рефокусуючі імпульси, що використовуються в ядерному магнітному резонансі. Детальний опис можна знайти в роботі [A Quantum Engineer's Guide to Superconducting Qubits](https://arxiv.org/abs/1904.06560).

Оскільки прохід `PadDynamicalDecoupling` працює лише з запланованими схемами та використовує гейти, які не обов'язково є базисними гейтами нашого цільового пристрою, тобі також знадобляться проходи [`ALAPScheduleAnalysis`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.ALAPScheduleAnalysis) та [`BasisTranslator`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.BasisTranslator).

Отримай інформацію про `target` із `backend` і збережи назви операцій як `basis_gates`, оскільки `target` потрібно буде змінити, щоб додати інформацію про часові характеристики гейтів, які використовуються в динамічному розв'язуванні.

> **Note:** Симульований бекенд `FakeSherbrooke` з `qiskit_ibm_runtime` використовується у цих прикладах, але ти можеш спробувати будь-який реальний або симульований бекенд, сумісний з Qiskit. Твої результати можуть відрізнятися.

In [1]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

target = backend.target
basis_gates = list(target.operation_names)

Створи схему `efficient_su2` як приклад. Спочатку транспілюй схему до бекенду, оскільки імпульси динамічного розв'язування потрібно додавати після того, як схему було транспільовано та заплановано. Динамічне розв'язування найкраще працює, коли в квантових схемах є багато часу простою — тобто коли одні кубіти не використовуються, поки інші активні. Це саме той випадок у цій схемі, оскільки двокубітні гейти `ecr` застосовуються послідовно в цьому ансаці.

In [2]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.circuit.library import efficient_su2

qc = efficient_su2(12, entanglement="circular", reps=1)
pm = generate_preset_pass_manager(1, target=target, seed_transpiler=12345)
qc_t = pm.run(qc)
qc_t.draw("mpl", fold=-1, idle_wires=False)

<Image src="../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/9479a60c-d5d0-45c7-a93e-a2a488ba8985-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/9479a60c-d5d0-45c7-a93e-a2a488ba8985-0.svg)

Послідовність динамічного розв'язування — це серія гейтів, що в сукупності дають тотожну операцію і рівномірно розподілені в часі. Наприклад, почнемо зі створення простої послідовності XY4, що складається з чотирьох гейтів.

In [3]:
from qiskit.circuit.library import XGate, YGate

X = XGate()
Y = YGate()

dd_sequence = [X, Y, X, Y]

Через регулярний часовий розподіл послідовностей динамічного розв'язування, інформацію про `YGate` необхідно додати до `target`, оскільки він *не є* базисним гейтом, на відміну від `XGate`. Проте ми знаємо *апріорі*, що `YGate` має таку ж тривалість та похибку, що й `XGate`, тож можемо просто отримати ці властивості з `target` і додати їх для об'єктів `YGate`. Саме тому `basis_gates` були збережені окремо — ми додаємо інструкцію `YGate` до `target`, хоча вона не є фактичним базисним гейтом `ibm_fez`.

In [4]:
from qiskit.transpiler import InstructionProperties

y_gate_properties = {}
for qubit in range(target.num_qubits):
    y_gate_properties.update(
        {
            (qubit,): InstructionProperties(
                duration=target["x"][(qubit,)].duration,
                error=target["x"][(qubit,)].error,
            )
        }
    )

target.add_instruction(YGate(), y_gate_properties)

Схеми ансацу, такі як `efficient_su2`, є параметризованими, тому перед відправкою до бекенду до них необхідно прив'язати значення. Тут призначимо випадкові параметри.

In [5]:
import numpy as np

rng = np.random.default_rng(1234)
qc_t.assign_parameters(
    rng.uniform(-np.pi, np.pi, qc_t.num_parameters), inplace=True
)

Далі виконай власні проходи. Створи екземпляр `PassManager` з `ALAPScheduleAnalysis` і `PadDynamicalDecoupling`. Спочатку запусти `ALAPScheduleAnalysis`, щоб додати інформацію про часові характеристики квантової схеми — тільки після цього можна додавати рівномірно розподілені в часі послідовності динамічного розв'язування. Ці проходи запускаються на схемі за допомогою `.run()`.

In [6]:
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes.scheduling import (
    ALAPScheduleAnalysis,
    PadDynamicalDecoupling,
)

dd_pm = PassManager(
    [
        ALAPScheduleAnalysis(target=target),
        PadDynamicalDecoupling(target=target, dd_sequence=dd_sequence),
    ]
)
qc_dd = dd_pm.run(qc_t)

Використай інструмент візуалізації [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer), щоб побачити часові характеристики схеми та переконатися, що рівномірно розподілена послідовність об'єктів `XGate` і `YGate` присутня в схемі.

In [7]:
from qiskit.visualization import timeline_drawer

timeline_drawer(qc_dd, idle_wires=False, target=target)

<Image src="../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/7a552621-a96f-4bb8-ae9b-4ab5a65bbb64-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/7a552621-a96f-4bb8-ae9b-4ab5a65bbb64-0.svg)

Нарешті, оскільки `YGate` не є фактичним базисним гейтом нашого бекенду, застосуй прохід `BasisTranslator` вручну (це стандартний прохід, але він виконується до планування, тому його потрібно застосувати знову). Бібліотека еквівалентностей сесії — це бібліотека еквівалентностей схем, що дозволяє Transpiler розкладати схеми на базисні гейти; вона також задається як аргумент.

In [8]:
from qiskit.circuit.equivalence_library import (
    SessionEquivalenceLibrary as sel,
)
from qiskit.transpiler.passes import BasisTranslator

qc_dd = BasisTranslator(sel, basis_gates)(qc_dd)
qc_dd.draw("mpl", fold=-1, idle_wires=False)

<Image src="../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/f0f4b29d-1c95-47d2-a7ad-8e130eaff74a-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/f0f4b29d-1c95-47d2-a7ad-8e130eaff74a-0.svg)

Тепер об'єкти `YGate` відсутні в нашій схемі, а явна інформація про часові характеристики представлена у вигляді гейтів `Delay`. Ця транспільована схема з динамічним розв'язуванням готова до відправки на бекенд.

## Наступні кроки

> **Tip:** - Щоб дізнатися, як використовувати функцію `generate_preset_passmanager` замість написання власних проходів, почни з теми [Параметри транспіляції за замовчуванням та параметри конфігурації](defaults-and-configuration-options).
> - Спробуй посібник [Порівняння налаштувань Transpiler](/guides/circuit-transpilation-settings#compare-transpiler-settings).
> - Переглянь [документацію Transpile API](https://docs.quantum-computing.ibm.com/api/qiskit/transpiler).